In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import json
from tqdm.notebook import tqdm
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix
from datasets import load_dataset
import re
import random

In [ ]:
#from huggingface_hub import login
#login("HF_TOKEN")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = AutoModel.from_pretrained("Himanshu167/AI-Response-Comparer-v1.6",
                                       trust_remote_code=True,
                                      cache_dir="./models")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Himanshu167/AI-Response-Comparer-v1.6",
                                          trust_remote_code=True,
                                          use_fast=False,
                                          cache_dir="./models")

In [ ]:
def robust_tokenize(text, text_pair=None, tokenizer=None, max_length=None, padding=False):
    enc_a = tokenizer(text, add_special_tokens=False)['input_ids']
    
    if text_pair is not None:
        enc_b = tokenizer(text_pair, add_special_tokens=False)['input_ids']
        
        if max_length is not None:
            while len(enc_a) + len(enc_b) + 3 > max_length:
                if len(enc_a) > len(enc_b):
                    enc_a.pop()
                else:
                    enc_b.pop()
                    
        input_ids = [1] + enc_a + [2] + enc_b + [2]
        
        token_type_ids = [0] * (len(enc_a) + 2) + [1] * (len(enc_b) + 1)

    else:
        if max_length is not None:
            enc_a = enc_a[:max_length - 2]
            
        input_ids = [1] + enc_a + [2]
        token_type_ids = [0] * len(input_ids)

    attention_mask = [1] * len(input_ids)

    if padding and max_length is not None:
        pad_len = max_length - len(input_ids)
        if pad_len > 0:
            input_ids += [0] * pad_len
            attention_mask += [0] * pad_len
            token_type_ids += [0] * pad_len

    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long)
    }

In [ ]:
dataset_name = "Anthropic/hh-rlhf"

In [ ]:
dataset = load_dataset(dataset_name, cache_dir="./datasets")

In [ ]:
dataset

In [ ]:
(len(dataset["train"]["chosen"]) == len(dataset["train"]["rejected"])) and (len(dataset["test"]["chosen"]) == len(dataset["test"]["rejected"]))

In [ ]:
data = dataset["train"]["chosen"][0].replace("\n\nAssistant: ", "Split this(9177c768383eef80ea35565339b301fa):")
data = data.replace("\n\nHuman: ", "Split this(9177c768383eef80ea35565339b301fa):")
data = data.split("Split this(9177c768383eef80ea35565339b301fa):")[1:]

In [ ]:
data

In [ ]:
def formatting_text(row):
    data = {"chosen":"", "rejected":""}
    data["chosen"] = row["chosen"].replace("\n\nAssistant: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["chosen"] = data["chosen"].replace("\n\nHuman: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["chosen"] = data["chosen"].split("Split this(9177c768383eef80ea35565339b301fa):")[1:]

    data["rejected"] = row["rejected"].replace("\n\nAssistant: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["rejected"] = data["rejected"].replace("\n\nHuman: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["rejected"] = data["rejected"].split("Split this(9177c768383eef80ea35565339b301fa):")[1:]

    tokens = re.findall(r'\n\nHuman:|\n\nAssistant:', row["chosen"])
    
    valid_A = (
        tokens and
        tokens[0] == '\n\nHuman:' and
        tokens[-1] == '\n\nAssistant:' and
        all(tokens[i] != tokens[i+1] for i in range(len(tokens)-1))
    )

    tokens = re.findall(r'\n\nHuman:|\n\nAssistant:', row["rejected"])
    
    valid_B = (
        tokens and
        tokens[0] == '\n\nHuman:' and
        tokens[-1] == '\n\nAssistant:' and
        all(tokens[i] != tokens[i+1] for i in range(len(tokens)-1))
    )

    assert valid_A and valid_B, "Pair not forming of prompt and response. It should be even to be prompt and response both"
    assert len(data["chosen"]) == len(data["rejected"]), "Unequal chosen and rejected lengths"
    assert data["chosen"][::2] == data["rejected"][::2], "Non-same prompts"

    data["prompt"] = data["chosen"][::2]
    data["response_a"] = data["chosen"][1::2]
    data["response_b"] = data["rejected"][1::2]
    
    output = {}
    output["response_1"] = []
    output["response_2"] = []
    for index in range(len(data["prompt"])):
        prompt = data["prompt"][index]
        response_a = data["response_a"][index]
        response_b = data["response_b"][index]
        output["response_1"].append(f"""Question: {prompt} Answer: {response_a}""")
        output["response_2"].append(f"""Question: {prompt} Answer: {response_b}""")
    return output

def filter_dataset(row, max_seq_len):
    try:
        data = formatting_text(row)
    except Exception as e:
        return False
    length_1 = 0
    length_2 = 0
    for response_1 in data["response_1"]:
        temp_len = len(tokenizer(response_1)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_1 += temp_len
    for response_2 in data["response_2"]:
        temp_len = len(tokenizer(response_2)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_2 += temp_len
    if length_1 > 1100 or length_2 > 1100:
        return False
    return True

In [ ]:
#dataset_train = dataset["train"].select(range(100))
dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [ ]:
dataset_train = dataset_train.filter(filter_dataset, fn_kwargs={"max_seq_len": 512})
dataset_test = dataset_test.filter(filter_dataset, fn_kwargs={"max_seq_len": 512})

In [ ]:
dataset_train

In [ ]:
dataset_test

In [ ]:
dataset_train = dataset_train.shuffle(seed=42)
dataset_test = dataset_test.shuffle(seed=42)

In [ ]:
dataset_train["chosen"][0]

In [ ]:
def seperate_response_prompt(row):
    choice = random.choice([True, False])
    
    data = {"chosen":"", "rejected":""}
    data["chosen"] = row["chosen"].replace("\n\nAssistant: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["chosen"] = data["chosen"].replace("\n\nHuman: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["chosen"] = data["chosen"].split("Split this(9177c768383eef80ea35565339b301fa):")[1:]

    data["rejected"] = row["rejected"].replace("\n\nAssistant: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["rejected"] = data["rejected"].replace("\n\nHuman: ", "Split this(9177c768383eef80ea35565339b301fa):")
    data["rejected"] = data["rejected"].split("Split this(9177c768383eef80ea35565339b301fa):")[1:]

    if choice:
        return {
            "prompt": data["chosen"][::2],
            "response_a": data["chosen"][1::2],
            "response_b": data["rejected"][1::2],
            "winner_model_a":1,
            "winner_model_b":0,
            "winner_tie":0,
        }
    else:
        return {
            "prompt": data["chosen"][::2],
            "response_a": data["rejected"][1::2],
            "response_b": data["chosen"][1::2],
            "winner_model_a":0,
            "winner_model_b":1,
            "winner_tie":0,
        }

In [ ]:
dataset_train = dataset_train.map(seperate_response_prompt)
dataset_test = dataset_test.map(seperate_response_prompt)

In [ ]:
dataset_test_1 = dataset_test.to_pandas()[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]]

# Merger

In [ ]:
def safe_json(cell):
    try:
        if isinstance(cell, list):
            return cell
        return json.loads(cell)
    except Exception:
        return [cell] if cell else []        

# Adds extra tokens such as question and answer
def formatting_text(row):
    output = {}
    output["response_1"] = []
    output["response_2"] = []
    for index in range(len(row["prompt"])):
        prompt = row["prompt"][index]
        response_a = row["response_a"][index]
        response_b = row["response_b"][index]
        output["response_1"].append(f"""Question: {prompt} Answer: {response_a}""")
        output["response_2"].append(f"""Question: {prompt} Answer: {response_b}""")
    return output

def filter_dataset(row, tokenizer, max_seq_len):
    length_1 = 0
    length_2 = 0
    for response_1 in row["response_1"]:
        temp_len = len(tokenizer(response_1)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_1 += temp_len
    for response_2 in row["response_2"]:
        temp_len = len(tokenizer(response_2)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_2 += temp_len
    if length_1 > 1100 or length_2 > 1100:
        return False
    return True

def prepare(tokenizer, data_path="data.csv"):
    df = pd.read_csv(data_path, quotechar='"', engine='python')

    list_cols = ["prompt", "response_a", "response_b"]

    for col in list_cols:
        df[col] = df[col].apply(safe_json)

    formatted = df.apply(formatting_text, axis=1, result_type="expand")
    df[["response_1", "response_2"]] = formatted

    max_seq_len = 512

    tqdm.pandas()

    keep_or_not = df.progress_apply(lambda row: filter_dataset(row, tokenizer, max_seq_len), axis=1)
    df = df[keep_or_not]
    df = df.drop(columns=['id', 'model_a', 'model_b', 'response_1', 'response_2'])
    return df

In [ ]:
df = prepare(tokenizer, "./train.csv")
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
dataset_test_2 = test

In [ ]:
len(train), len(test)

In [ ]:
final_train_dataset = pd.concat([dataset_train.to_pandas()[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]], train[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]]])
final_test_dataset = pd.concat([dataset_test.to_pandas()[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]], test[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]]])

In [ ]:
final_train_dataset.head()

In [ ]:
final_test_dataset.tail()

In [ ]:
final_train_dataset = final_train_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
final_test_dataset = final_test_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Merger Part 2

In [ ]:
dataset_3 = load_dataset("lmsys/chatbot_arena_conversations", cache_dir="./datasets")

In [ ]:
dataset_3

In [ ]:
dataset_3["train"][1]

In [ ]:
def formatting_text(row):
    assert len(row["conversation_a"])==len(row["conversation_b"]), "Conversation should be of same length to compare"
    assert row["conversation_a"][::2]==row["conversation_b"][::2], "Prompt should be same"
    assert len(row["conversation_a"])%2==0, "Q-A pair should form"
    
    prompt = row["conversation_a"][::2]
    response_a = row["conversation_a"][1::2]
    response_b = row["conversation_b"][1::2]
    
    assert len(prompt)==len(response_a), "All prompt should have assistant response."
    assert len(prompt)==len(response_b), "All prompt should have assistant response."

    data = {}
    data["prompt"] = []
    data["response_a"] = []
    data["response_b"] = []

    for i in range(len(prompt)):
        assert prompt[i]["role"]=="user", "Prompt should be by user"
        assert response_a[i]["role"]=="assistant", "Answer should be by assistant"
        assert response_b[i]["role"]=="assistant", "Answer should be by assistant"
        data["prompt"].append(prompt[i]["content"])
        data["response_a"].append(response_a[i]["content"])
        data["response_b"].append(response_b[i]["content"])

    output = {}
    output["response_1"] = []
    output["response_2"] = []
    
    for index in range(len(data["prompt"])):
        prompt = data["prompt"][index]
        response_a = data["response_a"][index]
        response_b = data["response_b"][index]
        output["response_1"].append(f"""Question: {prompt} Answer: {response_a}""")
        output["response_2"].append(f"""Question: {prompt} Answer: {response_b}""")
    return output

def filter_dataset(row, max_seq_len):
    data = formatting_text(row)
    length_1 = 0
    length_2 = 0
    for response_1 in data["response_1"]:
        temp_len = len(tokenizer(response_1)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_1 += temp_len
    for response_2 in data["response_2"]:
        temp_len = len(tokenizer(response_2)["input_ids"])
        if temp_len>500:
            return False
        if temp_len%max_seq_len!=0:
            temp_len = max_seq_len + (temp_len - (temp_len%max_seq_len))
        length_2 += temp_len
    if length_1 > 1100 or length_2 > 1100:
        return False
    return True

In [ ]:
dataset_3_filtered = dataset_3.filter(filter_dataset, fn_kwargs={"max_seq_len": 512})

In [ ]:
dataset_3_filtered

In [ ]:
set(dataset_3_filtered["train"]["winner"])

In [ ]:
def seperate_response_prompt(row):
    prompt = row["conversation_a"][::2]
    response_a = row["conversation_a"][1::2]
    response_b = row["conversation_b"][1::2]

    data = {}
    data["prompt"] = []
    data["response_a"] = []
    data["response_b"] = []

    for i in range(len(prompt)):
        data["prompt"].append(prompt[i]["content"])
        data["response_a"].append(response_a[i]["content"])
        data["response_b"].append(response_b[i]["content"])

    return {
        "prompt": data["prompt"],
        "response_a": data["response_a"],
        "response_b": data["response_b"],
        "winner_model_a":int(row["winner"]=="model_a"),
        "winner_model_b":int(row["winner"]=="model_b"),
        "winner_tie":int(row["winner"].startswith("tie")),
    }

In [ ]:
dataset_3_mapped = dataset_3_filtered.map(seperate_response_prompt)

In [ ]:
dataset_3_mapped["train"][0]

In [ ]:
dataset_3_df = dataset_3_mapped["train"].to_pandas()[["prompt","response_a","response_b","winner_model_a","winner_model_b","winner_tie"]]

In [ ]:
dataset_3_df.head()

In [ ]:
(dataset_3_df["winner_model_a"] + dataset_3_df["winner_model_b"] + dataset_3_df["winner_tie"]).unique()

In [ ]:
dataset_3_train, dataset_3_test = train_test_split(dataset_3_df, test_size=0.2, random_state=42)

In [ ]:
dataset_test_3 = dataset_3_test

In [ ]:
len(final_train_dataset), len(final_test_dataset)

In [ ]:
final_train_dataset = pd.concat([final_train_dataset, dataset_3_train])
final_test_dataset = pd.concat([final_test_dataset, dataset_3_test])

In [ ]:
len(final_train_dataset), len(final_test_dataset)

In [ ]:
final_train_dataset.head()

In [ ]:
final_test_dataset.tail()

In [ ]:
final_train_dataset = final_train_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
final_test_dataset = final_test_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
def counting_empty(row):
    if len(row["prompt"])==0 or len(row["response_a"])==0 or len(row["response_b"])==0:
        return 1
    return 0

In [ ]:
empty_count = final_train_dataset.apply(counting_empty, axis=1).sum()
empty_count += final_test_dataset.apply(counting_empty, axis=1).sum()

In [ ]:
empty_count

# Main

In [ ]:
class ScorerDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.input_ids_1 = []
        self.input_ids_2 = []
        for idx, row in tqdm(data.iterrows(), total=len(data), desc="Tokenizing"):
            prompt = [prompt1 if prompt1 else "" for prompt1 in row["prompt"]]
            responses_1 = [response if response else "" for response in row["response_a"]]
            responses_2 = [response if response else "" for response in row["response_b"]]

            assert(len(prompt)==len(responses_1))
            assert(len(prompt)==len(responses_2))
            enc1 = {'input_ids': [],
                    'attention_mask': [],
                    'token_type_ids': []}
            enc2 = {'input_ids': [],
                    'attention_mask': [],
                    'token_type_ids': []}
            for i in range(len(prompt)):
                enc1_1 = robust_tokenize(prompt[i], responses_1[i], tokenizer=tokenizer, max_length=512, padding=True)
                enc2_1 = robust_tokenize(prompt[i], responses_2[i], tokenizer=tokenizer, max_length=512, padding=True)
                enc1['input_ids'].append(enc1_1['input_ids'])
                enc1['attention_mask'].append(enc1_1['attention_mask'])
                enc1['token_type_ids'].append(enc1_1['token_type_ids'])

                enc2['input_ids'].append(enc2_1['input_ids'])
                enc2['attention_mask'].append(enc2_1['attention_mask'])
                enc2['token_type_ids'].append(enc2_1['token_type_ids'])

            enc1['input_ids'] = torch.stack(enc1['input_ids'])
            enc1['attention_mask'] = torch.stack(enc1['attention_mask'])
            enc1['token_type_ids'] = torch.stack(enc1['token_type_ids'])

            enc2['input_ids'] = torch.stack(enc2['input_ids'])
            enc2['attention_mask'] = torch.stack(enc2['attention_mask'])
            enc2['token_type_ids'] = torch.stack(enc2['token_type_ids'])
            
            self.input_ids_1.append(enc1)
            self.input_ids_2.append(enc2)
    
        self.labels = torch.tensor(
            data[["winner_model_a", "winner_model_b", "winner_tie"]].values,
            dtype=torch.float
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.input_ids_1[idx],
            self.input_ids_2[idx],
            self.labels[idx]
        )

def create_dataset(tokenizer, data_path="data.csv"):
    df = prepare(tokenizer, data_path)
    train, test = train_test_split(df, test_size=0.2, random_state=42)
    train_dataset = ScorerDataset(train, tokenizer)
    test_dataset = ScorerDataset(test, tokenizer)
    return train_dataset, test_dataset

In [ ]:
final_dataset_test_1 = ScorerDataset(dataset_test_1, tokenizer)
final_dataset_test_2 = ScorerDataset(dataset_test_2, tokenizer)
final_dataset_test_3 = ScorerDataset(dataset_test_3, tokenizer)

In [ ]:
test_loader_1 = DataLoader(
    final_dataset_test_1,
    batch_size=6,
    num_workers=0,
    collate_fn=lambda batch: (
        {
            "input_ids": [x[0]["input_ids"] for x in batch],
            "attention_mask": [x[0]["attention_mask"] for x in batch],
        },
        {
            "input_ids": [x[1]["input_ids"] for x in batch],
            "attention_mask": [x[1]["attention_mask"] for x in batch],
        },
        torch.stack([x[2] for x in batch])
    )
)

test_loader_2 = DataLoader(
    final_dataset_test_2,
    batch_size=6,
    num_workers=0,
    collate_fn=lambda batch: (
        {
            "input_ids": [x[0]["input_ids"] for x in batch],
            "attention_mask": [x[0]["attention_mask"] for x in batch],
        },
        {
            "input_ids": [x[1]["input_ids"] for x in batch],
            "attention_mask": [x[1]["attention_mask"] for x in batch],
        },
        torch.stack([x[2] for x in batch])
    )
)

test_loader_3 = DataLoader(
    final_dataset_test_3,
    batch_size=6,
    num_workers=0,
    collate_fn=lambda batch: (
        {
            "input_ids": [x[0]["input_ids"] for x in batch],
            "attention_mask": [x[0]["attention_mask"] for x in batch],
        },
        {
            "input_ids": [x[1]["input_ids"] for x in batch],
            "attention_mask": [x[1]["attention_mask"] for x in batch],
        },
        torch.stack([x[2] for x in batch])
    )
)

In [ ]:
model = model.to(device)

In [ ]:
def eval_one_epoch(test_loader):
    global output_array, label_array, original_output_array
    output_array = []
    label_array = []
    original_output_array = []
    log_steps = 1000
    zero_tensor = torch.tensor(0.0, dtype=torch.float, device=device)
    device_type = "cuda" if device.type == "cuda" else "cpu"
    is_cuda = (device_type == "cuda")
    
    correct_count = 0

    model.eval()

    for i, (input_1_og, input_2_og, labels) in enumerate(tqdm(test_loader, total=len(test_loader), desc="Evaluating")):        
        input_1 = {k: v for k, v in input_1_og.items()}
        input_2 = {k: v for k, v in input_2_og.items()}

        labels = labels.to(device)
        chunk_sizes = torch.tensor([len(chunk) for chunk in input_1["input_ids"]], dtype=torch.long, device=device)
        input_1["input_ids"] = torch.cat(input_1["input_ids"], dim=0).to(device)
        input_2["input_ids"] = torch.cat(input_2["input_ids"], dim=0).to(device)

        input_1["attention_mask"] = torch.cat(input_1["attention_mask"], dim=0).to(device)
        input_2["attention_mask"] = torch.cat(input_2["attention_mask"], dim=0).to(device)

        with torch.no_grad():
            outputs = model(input_1, input_2, chunk_sizes)
        
        [original_output_array.append(output.detach().cpu().numpy()) for output in outputs]
        outputs = outputs.argmax(dim=1)        
        
        [output_array.append(output.detach().cpu().numpy()) for output in outputs]
        
        labels = labels.argmax(dim=1)
        [label_array.append(label.detach().cpu().numpy()) for label in labels]
        
        correct_count += (outputs == labels).sum().item()
            
        if (i+1)%log_steps==0:
            print(f"Batch {i+1} Accuracy:", correct_count*100/(i*len(labels)))

    output_array = np.array(output_array)
    label_array = np.array(label_array)
    return correct_count*100/len(test_loader.dataset)

In [ ]:
accuracy = eval_one_epoch(test_loader_1)

In [ ]:
accuracy_1 = accuracy
original_output_array_1 = original_output_array.copy()
label_array_1, output_array_1 = label_array.copy(), output_array.copy()

In [ ]:
print(f"Accuracy: {accuracy}")
    
cm = confusion_matrix(label_array, output_array)
print("Confusion matrix", cm)

In [ ]:
accuracy = eval_one_epoch(test_loader_2)

In [ ]:
accuracy_2 = accuracy
original_output_array_2 = original_output_array.copy()
label_array_2, output_array_2 = label_array.copy(), output_array.copy()

In [ ]:
print(f"Accuracy: {accuracy}")
    
cm = confusion_matrix(label_array, output_array)
print("Confusion matrix", cm)

In [ ]:
accuracy = eval_one_epoch(test_loader_3)

In [ ]:
accuracy_3 = accuracy
original_output_array_3 = original_output_array.copy()
label_array_3, output_array_3 = label_array.copy(), output_array.copy()

In [ ]:
print(f"Accuracy: {accuracy}")
    
cm = confusion_matrix(label_array, output_array)
print("Confusion matrix", cm)